# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "."

In [3]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [4]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /workspaces/Code/notebooks/dl/assignments/assignment-1


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [5]:
from Tools.download import download_mini

download_mini(data_dir="_data")

Step 1 / 2  —  Mini dataset (SQuAD + GloVe)
  [skip] Mini dataset already present in _data/.

Step 2 / 2  —  spaCy language model
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

Mini dataset download complete.


---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [6]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:04<00:00, 34.63it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:01<00:00, 27.82it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:08<00:00, 3624.95it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:02<00:00, 3716.37it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data/train.npz',
 'dev_record_file': '_data/dev.npz',
 'word_emb_file': '_data/word_emb.json',
 'char_emb_file': '_data/char_emb.json',
 'train_eval_file': '_data/train_eval.json',
 'dev_eval_file': '_data/dev_eval.json',
 'word2idx_file': '_data/word2idx.json',
 'char2idx_file': '_data/char2idx.json',
 'dev_meta_file': '_data/dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [1]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 200, #60000,
    batch_size = 4, #8,
    seed       = 42,

    # ── updated recipe: adam, cosine scheduler, NLL loss ───────────────────────
    # *EXP-001
    # *hypothesis: Adam will be more stable than SGD on the repaired implementation
    # *intervention: changed optimizer_name from "sgd" to "adam"
    # *control: kept the training recipe and evaluation setup fixed
    # *result: Adam avoided NaN loss, but losses remained extremely large and QA performance stayed poor
    optimizer_name = "adam",
    # *FIX-I-003
    # *change: 'scheduler_name = "none"'
    # *rationale: replaces an unsupported scheduler name with one that TrainTools/train.py can actually construct
    # *EXP-002                                                                                                                                                                                                        
    # *hypothesis: a warmup phase followed by cosine decay will improve early optimization stability
    # *intervention: changed the scheduler to the warmup-plus-cosine lambda schedule and enabled warmup_steps=20
    # *control: kept the optimizer, dataset, batch size, num_steps, seed, loss, and evaluation setup fixed
    # *result: losses decreased substantially relative to the previous Adam baseline, but QA performance remained weak
    scheduler_name  = "lambda",
    warmup_steps    = 20,
    # *EXP-003
    # *hypothesis: lowering the learning rate under the warmup-plus-cosine schedule will improve training stability
    # *intervention: reduced learning_rate to 1e-4
    # *control: kept the optimizer, scheduler design, dataset, batch size, num_steps, seed, loss, and evaluation setup fixed
    # *result: losses dropped to a numerically reasonable range and EM improved, so 1e-4 was adopted for later experiments
    learning_rate   = 1e-4,
    # *EXP-005
    # *hypothesis: modest weight decay changes will affect stability and/or generalization under the repaired setup
    # *intervention: compared alternative weight_decay values, including 0.0, 1e-3, and 1e-5
    # *control: kept the optimizer, scheduler, learning rate, dataset, batch size, num_steps, seed, loss, and evaluation setup fixed
    # *result: weight decay had only a modest effect overall, but 1e-3 gave the best F1 and was adopted for later experiments
    weight_decay    = 1e-3,
    loss_name       = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

Optimizer LR at init: 0.0001


100%|██████████| 200/200 [10:30<00:00,  3.15s/it]


STEP      200  loss 47.725302



100%|██████████| 150/150 [01:29<00:00,  1.68it/s]


VALID(train) loss 13.708825  F1 6.640820  EM 0.000000



100%|██████████| 150/150 [01:30<00:00,  1.66it/s]


TEST        loss 13.785058  F1 7.219927  EM 0.000000

Learning rate: [1e-10]
Training finished.  Best F1: 7.2199  Best EM: 0.0000
Best F1: 7.2199  |  Best EM: 0.0000


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [2]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [24:43<00:00,  1.13s/it]


TEST  loss 15.436168  F1 7.861785  EM 0.019111
F1: 7.8618  |  EM: 0.0191  |  Loss: 15.436168
